# Pain-rating correlations across trials

This notebook contains one corrected analysis of whether rating-temperature correlations change across the 12 trial positions. Because trials are repeated within participants, the analysis retains `participant_id`, Fisher-transforms the correlations, includes participant fixed effects, and clusters inference by participant.

No stimulus-curve or skin-patch adjustments are included. Stimulus order was randomized and patch order was counterbalanced, so neither variable is required to estimate the overall serial-position association under the study protocol.

In [1]:
from pathlib import Path
import os

configured_root = os.getenv("PAIN_MEASUREMENT_ROOT")
candidates = [Path.cwd(), Path.cwd().parent]
if configured_root:
    candidates.append(Path(configured_root))

PROJECT_ROOT = next(
    path
    for path in candidates
    if (path / "src").is_dir() and (path / "pain-measurement.duckdb").exists()
)
os.chdir(PROJECT_ROOT)
PROJECT_ROOT

PosixPath('/Users/visser/drive/PhD/Code/pain-measurement')

In [2]:
import altair as alt
import numpy as np
import pandas as pd
import polars as pl
import statsmodels.formula.api as smf

from src.data.database_manager import DatabaseManager

## Trial-level correlations

Each row below is one participant's Pearson correlation between pain ratings and temperature within one trial.

In [3]:
with DatabaseManager() as db:
    df = db.get_trials("Explore_Data", exclude_problematic=True)

corr_by_trial = (
    df.group_by("participant_id", "trial_id", "trial_number")
    .agg(pl.corr("rating", "temperature").alias("correlation"))
    .sort("participant_id", "trial_number")
)

analysis_df = corr_by_trial.to_pandas()
eps = np.finfo(float).eps
analysis_df["fisher_z"] = np.arctanh(
    analysis_df["correlation"].clip(-1 + eps, 1 - eps)
)
analysis_df["trial_c"] = analysis_df["trial_number"] - 6.5
analysis_df.head()

,participant_id,trial_id,trial_number,correlation,fisher_z,trial_c
0,1,1,1,0.677113,0.823763,-5.5
1,1,2,2,0.670486,0.811625,-4.5
2,1,3,3,0.650548,0.776249,-3.5
3,1,4,4,0.661172,0.794892,-2.5
4,1,5,5,0.784558,1.057118,-1.5


In [4]:
participant_trial_counts = analysis_df.groupby("participant_id").size()
trial_group_sizes = analysis_df.groupby("trial_number").size()

data_summary = pd.Series(
    {
        "valid_trials": len(analysis_df),
        "participants": analysis_df["participant_id"].nunique(),
        "complete_participants": int((participant_trial_counts == 12).sum()),
        "observations_per_trial_min": int(trial_group_sizes.min()),
        "observations_per_trial_max": int(trial_group_sizes.max()),
    },
    name="value",
)

assert data_summary["valid_trials"] == 471
assert data_summary["participants"] == 42
assert analysis_df[["participant_id", "trial_number"]].duplicated().sum() == 0
assert analysis_df["correlation"].between(-1, 1).all()
data_summary.to_frame()

,value
valid_trials,471
participants,42
complete_participants,34
observations_per_trial_min,36
observations_per_trial_max,42


## Supplementary figure

Boxplots show the participant-level distribution at each trial position. Black points and the connecting line show the unadjusted mean correlation.

In [5]:
boxplots = (
    alt.Chart(analysis_df)
    .mark_boxplot(color="#52b788")
    .encode(
        x=alt.X(
            "trial_number:O",
            title="Trial number",
            axis=alt.Axis(labelAngle=0),
        ),
        y=alt.Y(
            "correlation:Q",
            title="Correlation between pain rating and temperature",
            scale=alt.Scale(domain=[0, 1]),
        ),
    )
)

mean_line = (
    alt.Chart(analysis_df)
    .transform_aggregate(
        mean_correlation="mean(correlation)",
        groupby=["trial_number"],
    )
    .mark_line(color="black", point={"filled": True, "size": 35})
    .encode(
        x=alt.X("trial_number:O", axis=alt.Axis(labelAngle=0)),
        y=alt.Y("mean_correlation:Q"),
        tooltip=[
            alt.Tooltip("trial_number:O", title="Trial"),
            alt.Tooltip("mean_correlation:Q", title="Mean r", format=".3f"),
        ],
    )
)

chart = (
    (boxplots + mean_line)
    .properties(width=600, height=300)
    .configure_axis(titleFontWeight="normal", labelFontWeight="normal")
)
chart

alt.LayerChart(...)

## Within-participant trial-position analysis

The model estimates the linear change in Fisher-transformed correlation per trial. Participant fixed effects remove all time-invariant between-participant differences. CR1 standard errors are clustered by participant, with inference based on 41 participant-cluster degrees of freedom.

In [6]:
model = smf.ols(
    "fisher_z ~ trial_c + C(participant_id)",
    data=analysis_df,
).fit(
    cov_type="cluster",
    cov_kwds={
        "groups": analysis_df["participant_id"],
        "use_correction": True,
        "df_correction": True,
    },
    use_t=True,
)

ci_low, ci_high = model.conf_int().loc["trial_c"]
slope = model.params["trial_c"]
total_change_z = 11 * slope
mean_z = analysis_df["fisher_z"].mean()
approximate_change_r = np.tanh(mean_z + total_change_z / 2) - np.tanh(
    mean_z - total_change_z / 2
)

trial_result = pd.Series(
    {
        "fisher_z_change_per_trial": slope,
        "clustered_standard_error": model.bse["trial_c"],
        "t": model.tvalues["trial_c"],
        "df": model.df_resid_inference,
        "p": model.pvalues["trial_c"],
        "ci_low": ci_low,
        "ci_high": ci_high,
        "trial_1_to_12_change_fisher_z": total_change_z,
        "approximate_trial_1_to_12_change_r": approximate_change_r,
    },
    name="value",
)
trial_result.to_frame()

,value
fisher_z_change_per_trial,0.005066
clustered_standard_error,0.002439
t,2.077252
df,41.000000
p,0.044083
ci_low,0.000141
ci_high,0.009992
trial_1_to_12_change_fisher_z,0.055729
approximate_trial_1_to_12_change_r,0.025930


In [7]:
print(
    f"Fisher-z change per trial = {slope:.4f} "
    f"(95% CI {ci_low:.4f} to {ci_high:.4f}), "
    f"t({int(model.df_resid_inference)}) = {model.tvalues['trial_c']:.2f}, "
    f"p = {model.pvalues['trial_c']:.3f}.\n"
    f"Estimated trial-1-to-12 change = {total_change_z:.4f} Fisher-z units, "
    f"approximately {approximate_change_r:.3f} on the raw-r scale around "
    "the observed mean.\n\n"
    "Trial position measures serial order; it does not identify a specific "
    "learning mechanism or demonstrate practical equivalence."
)

Fisher-z change per trial = 0.0051 (95% CI 0.0001 to 0.0100), t(41) = 2.08, p = 0.044.
Estimated trial-1-to-12 change = 0.0557 Fisher-z units, approximately 0.026 on the raw-r scale around the observed mean.

Trial position measures serial order; it does not identify a specific learning mechanism or demonstrate practical equivalence.


## Suggested reporting

Pain-temperature correlations remained high across all trials. A participant-clustered within-subject analysis of Fisher-z-transformed correlations indicated a small positive association with trial order (βz = 0.0051 per trial, 95% CI [0.0001, 0.0100], p = .044), corresponding to an estimated increase in r of approximately .026 from trial 1 to trial 12. Trial order reflects time on task and does not isolate a specific learning effect.